# Week 8 — Bayesian Optimization

Generate optimized recommendations for Week 8 using utility modules.

## Setup

In [ ]:
import numpy as np
import warnings
import sys
import importlib
sys.path.append('..')  # Add parent directory to path

# Import utility modules (reload to pick up any code changes)
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import propose_next_point, fit_gp, get_strategy

from utils.data_utils import (
    load_week_data,
    save_week_data,
    combine_with_week_results, 
    print_data_summary
)

## 1. Load Week 7 Data

Load the combined data from previous weeks

In [ ]:
# Load Week 7 clean data
inputs, outputs = load_week_data("../week 7/week7_clean_data.npz")
print_data_summary(inputs, outputs, "Week 7 Data")

## 2. Add Week 7 Results

In [ ]:
# Week 7 submitted points
week7_inputs = {
    1: np.array([0.418000, 0.415000]),
    2: np.array([0.678994, 0.095968]),
    3: np.array([0.346863, 0.672420, 0.439172]),
    4: np.array([0.413690, 0.367443, 0.360391, 0.413441]),
    5: np.array([0.000000, 0.000000, 1.000000, 1.000000]),
    6: np.array([0.755469, 0.270580, 0.644099, 0.672228, 0.162862]),
    7: np.array([0.000000, 0.317263, 0.707844, 0.246481, 0.405689, 0.758028]),
    8: np.array([0.107475, 0.120302, 0.020000, 0.217071, 1.000000, 0.099881, 0.180000, 0.998620])
}

# Week 7 outputs (received from black box)
week7_outputs = {
    1: 0.8787054931719586,
    2: 0.5824221881927588,
    3: -0.007685783181087912,
    4: 0.7243142534585663,
    5: 1616.6425,
    6: -0.6061521376508996,
    7: 1.8617337531726526,
    8: 9.761422753904
}

# Combine with Week 7 results
inputs, outputs = combine_with_week_results(inputs, outputs, week7_inputs, week7_outputs)
print_data_summary(inputs, outputs, "After Week 7 Results")

In [ ]:
# Save combined data for Week 8
save_week_data(inputs, outputs, "week8_clean_data.npz")

## 3. Week 7 Results Analysis

Evaluate which strategies worked and which failed to inform Week 8 approach.

In [ ]:
# Week 7 results analysis
print("=" * 70)
print("WEEK 7 RESULTS ANALYSIS")
print("=" * 70)

# Best values before Week 7 query
best_before_w7 = {}
for fid in range(1, 9):
    best_before_w7[fid] = np.max(outputs[fid][:-1])

print(f"\n{'F':>2} {'Dims':>4} {'Best Before W7':>14} {'W7 Query':>14} {'New Best':>14} {'Status'}")
print("-" * 70)

improved = 0
for fid in range(1, 9):
    dim = inputs[fid].shape[1]
    prev_best = best_before_w7[fid]
    w7_val = week7_outputs[fid]
    new_best = np.max(outputs[fid])
    
    if w7_val >= prev_best:
        status = "NEW BEST"
        improved += 1
    else:
        status = f"miss (best still {prev_best:.4f})"
    
    print(f"{fid:>2} {dim:>3}D {prev_best:>14.4f} {w7_val:>14.4f} {new_best:>14.4f}   {status}")

print(f"\nWeek 7 hit rate: {improved}/8 functions improved")
print("=" * 70)

In [ ]:
# Detailed Week 7 strategy evaluation
print("=" * 70)
print("WEEK 7 STRATEGY EVALUATION — What worked, what didn't")
print("=" * 70)

strategies_w7 = {
    1: ("Manual dim2+0.005", "NEW BEST +0.173 (0.706→0.879). Huge jump. 5 consecutive improvements. Micro-nudge on dim2 validated."),
    2: ("EI wide xi=0.5", "MISS 0.582 vs best 0.614. Wide exploration didn't find the second peak (peer's 0.829). dim2 shifted too low (0.096)."),
    3: ("Manual dim1-0.001", "MISS -0.0077 vs best -0.0056. Slightly worse. dim1 decrease overshot — W4's 0.348 was closer to optimum than 0.347."),
    4: ("Manual dim3-0.005", "NEW BEST +0.014 (0.710→0.724). dim3 decrease trend confirmed. Reverting from W6's EI failure was correct."),
    5: ("EXPLORE [0,0,1,1]", "INFO ONLY 1617 (best remains 8662). Expected. Learned low dim1&2 + high dim3&4 gives ~1600-1700 range."),
    6: ("Manual dim2-0.005", "MISS -0.606 vs best -0.521. Worse than W6. dim2 decrease direction is wrong — should go back up."),
    7: ("Manual dim2-0.005", "NEW BEST +0.008 (1.854→1.862). Small but consistent gain. dim2 sweet spot narrowing: 0.317-0.322."),
    8: ("Manual dim4+0.01", "MISS 9.761 vs best 9.763. Near-miss by 0.001. dim4 at 0.217 slightly past optimum (W4's 0.207 was better).")
}

for fid in range(1, 9):
    strategy, evaluation = strategies_w7[fid]
    print(f"\nF{fid}: {strategy}")
    print(f"  → {evaluation}")

print(f"\n{'=' * 70}")
print("SUMMARY: 3/8 new bests (F1, F4, F7) — all from manual single-dim nudges")
print("Manual nudges: 3 wins, 3 misses | EI wide: 0 wins, 1 miss | Explore: 1 info gain")
print("=" * 70)

In [ ]:
# Full history tracker — best score per function per week
print("=" * 70)
print("CUMULATIVE BEST SCORES ACROSS ALL WEEKS")
print("=" * 70)

# Track best at each week boundary
# Each function has initial data (10 pts) + 7 weekly queries = 17 points now
print(f"\n{'F':>2} {'W1':>10} {'W2':>10} {'W3':>10} {'W4':>10} {'W5':>10} {'W6':>10} {'W7':>10}")
print("-" * 80)

week_results = {
    1: [0.0979, 3.1e-39, 0.3255, 0.4147, 0.6114, 0.7062, 0.8787],
    2: [0.5567, 0.6138, 0.0480, 0.6050, 0.5471, 0.5725, 0.5824],
    3: [-0.0593, -0.0499, -0.1750, -0.0056, -0.0701, -0.0079, -0.0077],
    4: [-4.4163, 0.3523, 0.4226, 0.6723, 0.7101, 0.4657, 0.7243],
    5: [1231.61, 1688.07, 7599.50, 8662.48, 8290.38, 8643.15, 1616.64],
    6: [-0.5920, -0.5210, -1.0560, -0.5902, -0.9899, -0.5207, -0.6062],
    7: [1.3646, 1.7845, 1.3720, 1.7718, 1.4720, 1.8536, 1.8617],
    8: [9.5863, 9.6493, 9.6972, 9.7627, 9.7274, 9.7402, 9.7614]
}

for fid in range(1, 9):
    running_best = []
    current_best = float('-inf')
    for val in week_results[fid]:
        current_best = max(current_best, val)
        running_best.append(current_best)
    
    vals = ' '.join(f'{v:>10.4f}' for v in running_best)
    print(f"{fid:>2} {vals}")

print(f"\n{'F':>2} {'Overall Best':>14} {'Best Week':>10}")
print("-" * 30)
for fid in range(1, 9):
    best_val = max(week_results[fid])
    best_week = week_results[fid].index(best_val) + 1
    print(f"{fid:>2} {best_val:>14.4f} {'W' + str(best_week):>10}")
print("=" * 70)

## 4. Sensitivity Analysis

Updated sensitivity analysis with Week 7 data to inform Week 8 strategies.

In [ ]:
from utils.sensitivity import sensitivity_analysis

for func_id in range(1, 9):
    sensitivity_analysis(func_id, inputs[func_id], outputs[func_id])

## 5. Week 8 Strategy Design

Strategies based on 7-week performance history, updated sensitivity analysis, GP kernel diagnostics, and failed strategy elimination.

### Week 7 lessons learned:
- **Manual dim2+0.005 on F1 = massive win** — 0.706→0.879, biggest single-week gain. Alternating dims works.
- **F4 manual dim3 decrease confirmed** — 0.710→0.724, reverting from W6's EI failure was correct.
- **F7 dim2-0.005 keeps working** — 1.854→1.862, sweet spot narrowing to 0.317-0.322.
- **F2 EI wide search failed** — 0.582, still can't find peer's 0.829 peak. Need different exploration strategy.
- **F6 dim2 decrease was wrong direction** — -0.606 vs -0.521. Also revealed **noise**: same input as W2 gave different output.
- **F8 near-miss** — dim4 at 0.217 slightly overshot vs W4's 0.207. Function nearly converged at 9.76.
- **F3 dim1 decrease overshot** — -0.0077 vs -0.0056. 0.348→0.347 was too far. Try opposite direction.
- **F5 exploration complete** — [0,0,1,1]=1617 confirms [1,1,1,1]=8662 is the dominant corner.

### Cumulative failed strategies to avoid:
- **F1:** Large moves (W2: +0.27 on dim2 → 3.1e-39). Only micro-nudges work.
- **F2:** Micro-nudges near [0.7, 0.125] failed 5 straight weeks. EI wide (W7) also failed. All queries clustered in dim1=0.68-0.81 — never explored low dim1.
- **F3:** Wide exploration failed (W3, W5). Both dim2-0.005 (W6) and dim1-0.001 (W7) failed. Try opposite direction.
- **F4:** EI regressed (W6). Manual dim3 decrease works (W5, W7).
- **F5:** Any deviation from [1,1,1,1] loses. Explore queries for info only.
- **F6:** Moving dims 1/3/4/5 → disaster. dim2 decrease wrong (W7). Function has noise (W2=W7 input, different outputs).
- **F7:** dim2 > 0.34 fails (W5). dim2 < 0.27 fails (W4). Sweet spot: 0.31-0.32.
- **F8:** Multi-dim changes failed (W5, W6). dim4 overshoot at 0.217 (W7). Best point from W4 still holds.

### Strategy per function:

| F | Strategy | Rationale | Avoids |
|---|----------|-----------|--------|
| F1 | **Manual** dim1+0.005 → [0.423, 0.415] | Alternate to dim1 (W7 did dim2). W6's dim1+0.005 gave +0.095. Continue alternating. | Repeating W7's dim2 move |
| F2 | **Manual explore** [0.35, 0.50] — untested quadrant | All 7 queries had dim1 > 0.67. Never explored low dim1. Peer's 0.829 peak is somewhere unknown. Centre of unexplored region = max info gain with moderate risk. | Extreme corners (UCB), failed micro-nudges, failed EI wide |
| F3 | **Manual** dim1+0.001 → [0.348863, 0.672420, 0.439172] | W7 overshot at 0.347. W4 best at 0.348. Try 0.349 — slightly above W4's optimum. | W7's dim1 decrease, W6's dim2 decrease |
| F4 | **Manual** dim3-0.005 → [0.413690, 0.367443, 0.355391, 0.413441] | Two consecutive gains from dim3 decrease (W5: 0.710, W7: 0.724). Continue the trend. | EI automation |
| F5 | **Explore** [1, 0, 1, 1] — test dim2 effect | Best locked at 8662. [0,0,1,1]=1617 from W7. Test [1,0,1,1] to isolate dim2's contribution. | Wasting query on known optimum |
| F6 | **Manual** dim2+0.005 → [0.755469, 0.280580, 0.644099, 0.672228, 0.162862] | W7 proved decrease is wrong. W6's +0.005 gave best. Push further in + direction. Account for noise. | dim2 decrease (W7 failure), moving other dims |
| F7 | **Manual** dim2-0.003 → [0.000, 0.314263, 0.707844, 0.246481, 0.405689, 0.758028] | Smaller step to avoid overshooting. W7's -0.005 still improved. Tighten the search. | Large dim2 moves, multi-dim changes |
| F8 | **Manual** dim4+0.005 from W4 best → [0.107, 0.120, 0.020, 0.212, 1.0, 0.100, 0.180, 0.999] | W4's dim4=0.207 was best. W7's 0.217 slightly worse. Split the difference at 0.212. | Multi-dim changes, large dim4 moves |

In [ ]:
import warnings
import importlib
import utils.bayesian_optimization
importlib.reload(utils.bayesian_optimization)
from utils.bayesian_optimization import fit_gp, propose_next_point

week8_recommendations = {}

def get_best_point(fid):
    """Get the best observed point for a function"""
    best_idx = np.argmax(outputs[fid])
    return inputs[fid][best_idx].copy()

# ============================================================
# F1: Manual — dim1+0.005 (alternate from W7's dim2+0.005)
# W7's dim2+0.005 gave +0.173 (massive). W6 did dim1+0.005 (+0.095).
# Alternating dims: W5=dim2, W6=dim1, W7=dim2. Now dim1's turn.
# Avoids: repeating W7's exact dim2 move.
# ============================================================
best1 = get_best_point(1)  # [0.418, 0.415] → 0.8787
week8_recommendations[1] = np.array([best1[0] + 0.005, best1[1]])

# ============================================================
# F2: Manual exploratory — [0.35, 0.50] — untested quadrant
# All 7 queries clustered around dim1=0.68-0.81. We've NEVER tried
# low dim1. Peer found 0.829 at unknown location.
# UCB kappa=5.0 pushed to extreme corner [0.95, 0.90] — too risky.
# Instead, try [0.35, 0.50] — centre of the completely unexplored
# low-dim1 region. Moderate risk, maximum information gain.
# Avoids: extreme corners, repeating failed micro-nudges, failed EI.
# ============================================================
week8_recommendations[2] = np.array([0.350000, 0.500000])

# ============================================================
# F3: Manual — dim1+0.001 (reverse W7's overshoot)
# W4 best at dim1=0.348 → -0.0056. W7 at dim1=0.347 → -0.0077.
# Going lower was wrong. Try 0.349 — slightly above W4 optimum.
# Avoids: W7's decrease, W6's dim2 change, W3/W5 wide exploration.
# ============================================================
best3 = get_best_point(3)  # W4 best: [0.347863, 0.672420, 0.439172] → -0.0056
week8_recommendations[3] = np.array([best3[0] + 0.001, best3[1], best3[2]])

# ============================================================
# F4: Manual — dim3-0.005 (continue proven trend)
# W5: dim3=0.365→0.710. W7: dim3=0.360→0.724. Two consecutive
# improvements from dim3 decrease. Continue to 0.355.
# Avoids: EI automation (W6 disaster).
# ============================================================
best4 = get_best_point(4)  # W7 best: [0.413690, 0.367443, 0.360391, 0.413441] → 0.7243
week8_recommendations[4] = np.array([best4[0], best4[1], best4[2] - 0.005, best4[3]])

# ============================================================
# F5: Explore [1, 0, 1, 1] — another corner for information
# [1,1,1,1]=8662 locked in. [0,0,1,1]=1617 from W7.
# Test [1,0,1,1] to understand dim2's independent effect.
# Resubmitting [1,1,1,1] wastes a query — no new information.
# Avoids: wasting query on known optimum.
# ============================================================
week8_recommendations[5] = np.array([1.000000, 0.000000, 1.000000, 1.000000])

# ============================================================
# F6: Manual — dim2+0.005 → [0.755, 0.281, 0.644, 0.672, 0.163]
# W7 proved dim2 decrease is wrong (-0.606). W6's dim2=0.276 gave
# -0.521 (best). Try dim2=0.281 — push further in the + direction.
# NOTE: F6 shows noise — same W2 input gave -0.521, W7 gave -0.606.
# Avoids: dim2 decrease (W7 failure), moving other dims.
# ============================================================
best6 = get_best_point(6)  # W6 best: [0.755469, 0.275580, ...] → -0.5207
week8_recommendations[6] = np.array([
    best6[0],            # dim1 locked
    best6[1] + 0.005,    # dim2 +0.005 (0.276→0.281)
    best6[2],            # dim3 locked
    best6[3],            # dim4 LOCKED
    best6[4]             # dim5 LOCKED
])

# ============================================================
# F7: Manual — dim2-0.003 (smaller step, tighten search)
# W6: dim2=0.322→1.854. W7: dim2=0.317→1.862. Both improved.
# Smaller step of 0.003 to avoid overshooting (W4: dim2=0.271 failed).
# Avoids: large dim2 moves, multi-dim changes.
# ============================================================
best7 = get_best_point(7)  # W7 best: [0.0, 0.317263, ...] → 1.8617
week8_recommendations[7] = np.array([
    best7[0],            # dim1 locked at 0.0
    best7[1] - 0.003,    # dim2 -0.003 (0.317→0.314)
    best7[2],            # dim3 locked
    best7[3],            # dim4 locked
    best7[4],            # dim5 locked
    best7[5]             # dim6 locked
])

# ============================================================
# F8: Manual — dim4+0.005 from W4 best → split difference
# W4: dim4=0.207→9.763 (best). W7: dim4=0.217→9.761 (near-miss).
# Midpoint: 0.212. Try this to triangulate the dim4 optimum.
# Avoids: multi-dim changes (W5/W6 failures).
# ============================================================
best8_idx = np.argmax(outputs[8])
best8 = inputs[8][best8_idx].copy()  # W4 best point
week8_recommendations[8] = np.array([
    best8[0],            # dim1 locked
    best8[1],            # dim2 locked
    best8[2],            # dim3 locked
    best8[3] + 0.005,    # dim4 +0.005 from W4's 0.207 → 0.212
    best8[4],            # dim5 locked at 1.0
    best8[5],            # dim6 locked
    best8[6],            # dim7 locked
    best8[7]             # dim8 locked
])

# ============================================================
# Sanity check all recommendations
# ============================================================
print("Week 8 Recommendations")
print("=" * 80)
for fid in range(1, 9):
    X, y = inputs[fid], outputs[fid]
    rec = week8_recommendations[fid]
    
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        gp = fit_gp(X, y)
    pred, pred_std = gp.predict(rec.reshape(1, -1), return_std=True)
    dists = np.linalg.norm(X - rec, axis=1)
    min_dist = np.min(dists)
    best = np.max(y)
    
    strategy = {
        1: "Manual dim1+0.005 (alternate from W7)",
        2: "Manual explore [0.35, 0.50] — untested low-dim1 quadrant",
        3: "Manual dim1+0.001 (reverse W7 overshoot)",
        4: "Manual dim3-0.005 (continue proven trend)",
        5: "EXPLORE [1,0,1,1] — test dim2 effect",
        6: "Manual dim2+0.005 (reverse W7 failure)",
        7: "Manual dim2-0.003 (smaller step, tighten)",
        8: "Manual dim4+0.005 from W4 best (split difference)"
    }
    
    print(f"F{fid} ({X.shape[1]}D)  best={best:.4f}  pred={pred[0]:.4f}±{pred_std[0]:.4f}  dist={min_dist:.4f}  {strategy[fid]}")
    print(f"  point: {rec}")
print("=" * 80)

## 6. Submission Format

In [ ]:
# Submission format
print("=" * 70)
print("WEEK 8 SUBMISSION")
print("=" * 70)

for fid in range(1, 9):
    pt = week8_recommendations[fid]
    formatted = '-'.join(f'{x:.6f}' for x in pt)
    print(f"Function {fid}:\t{formatted}")